In [ ]:
pip install tf-keras

In [ ]:
pip install transformers==4.46.3

In [1]:
import torch
from torch import nn
from tqdm import tqdm
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2024-12-04 21:46:23.947079: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-04 21:46:24.326353: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-04 21:46:24.678591: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733348784.947546    5085 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733348785.025376    5085 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempt

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [83]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from torch.utils.data import Dataset
# from transformers import AutoTokenizer, AutoModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)  # Specify the number of classes

class TextDataset(Dataset):
    def __init__(self, texts, targets):
        self.texts = texts
        self.targets = targets
        print(self.targets[0])  # Используйте self.targets для доступа к целевым меткам

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text_tensor = self.texts[idx].clone().detach().to(device)
        
        # Измените создание target_tensor на FloatTensor
        target_tensor = torch.tensor(self.targets[idx], dtype=torch.float).to(device)  # Убедитесь, что это FloatTensor
        
        return text_tensor, target_tensor

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [84]:
# import torch
# from torch import nn

# class TextClassifier(nn.Module):
#     def __init__(self, bert_model):
#         super(TextClassifier, self).__init__()
#         self.bert = bert_model
#         self.dropout = nn.Dropout(0.2)
#         self.pooler = nn.Sequential(
#             nn.Linear(bert_model.config.hidden_size, bert_model.config.hidden_size),
#             nn.Tanh(),
#         )
#         self.classifier = nn.Linear(bert_model.config.hidden_size, 2)

#     def forward(self, input_ids, attention_mask):
#         outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         hidden_states = outputs.last_hidden_state
#         mean_pooling = torch.mean(hidden_states, dim=1)
#         pooled_output = self.pooler(mean_pooling)
#         pooled_output = self.dropout(pooled_output)
#         logits = self.classifier(pooled_output)
#         return logits

In [85]:
import pandas as pd

df = pd.read_csv('amazon_rev/test.csv', engine='python', on_bad_lines='skip')
df = df.dropna()

In [86]:
import re

df['2'] = df['2'].apply(lambda x: [1, 0] if x == 1 else [0, 1]) 

texts = df[df.columns[-1]].tolist()
targets = df['2'].tolist()

positive_samples = df[df['2'].apply(lambda x: x == [1, 0])].sample(n=10000, random_state=42)
negative_samples = df[df['2'].apply(lambda x: x == [0, 1])].sample(n=10000, random_state=42)

# Объединяем позитивные и негативные примеры
combined_samples = pd.concat([positive_samples, negative_samples])

# Перемешиваем данные
combined_samples = combined_samples.sample(frac=1, random_state=42).reset_index(drop=True)

# Проверяем размер нового DataFrame
print(combined_samples.shape)

# Извлекаем тексты и таргеты
texts = combined_samples[combined_samples.columns[-1]].tolist()
targets = combined_samples['2'].tolist()

# Кодируем тексты
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors='pt'
)

(20000, 3)


In [87]:
X_train, X_test, y_train, y_test = train_test_split(encodings['input_ids'], targets, test_size=0.2, random_state=42)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

from torch.optim.lr_scheduler import CosineAnnealingLR

num_epochs = 3

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = nn.BCEWithLogitsLoss()

[0, 1]
[1, 0]


In [88]:
def train(model, train_loader, optimizer, criterion, device, num_epochs=100):
    model.train()  
    for epoch in range(num_epochs):  
        total_loss = 0
        all_predictions = []
        all_targets = []
        
        for batch in tqdm(train_loader):
            input_ids, targets = batch
            attention_mask = (input_ids != 0).type(torch.int)
            
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            # Access logits instead of outputs directly
            logits = outputs.logits
            
            # Apply sigmoid to logits for probabilities
            probabilities = torch.sigmoid(logits)  
            
            # Calculate loss using logits
            loss = criterion(logits, targets) 
            total_loss += loss.item()
            
            # Save probabilities and targets
            all_predictions.append(probabilities.detach().cpu().numpy())
            all_targets.append(targets.detach().cpu().numpy())

            loss.backward()
            optimizer.step()

        # Combine all batches into one-dimensional arrays
        all_predictions = np.concatenate(all_predictions)
        all_targets = np.concatenate(all_targets)

        # Use probabilities for metric calculations
        accuracy_train = accuracy_score(all_targets, (all_predictions > 0.5).astype(float))
        f1_train = f1_score(all_targets, (all_predictions > 0.5).astype(float), average='macro')
        
        return f'Train Loss: {total_loss:.4f}, Train Accuracy: {accuracy_train:.4f}, Train F1 Score: {f1_train:.4f}'

def test(model, test_loader, criterion, device):
    model.eval()  
    total_loss = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():  
        for batch in tqdm(test_loader):
            input_ids, targets = batch
            attention_mask = (input_ids != 0).type(torch.int)
            
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            # Access logits instead of outputs directly
            logits = outputs.logits
            
            # Apply sigmoid to logits for probabilities
            probabilities = torch.sigmoid(logits)  
            
            # Calculate loss using logits
            loss = criterion(logits, targets) 
            total_loss += loss.item()

            # Save probabilities and targets
            all_predictions.append(probabilities.detach().cpu().numpy())
            all_targets.append(targets.detach().cpu().numpy())

    # Combine all batches into one-dimensional arrays
    all_predictions = np.concatenate(all_predictions)
    all_targets = np.concatenate(all_targets)

    # Use probabilities for metric calculations
    accuracy_test = accuracy_score(all_targets, (all_predictions > 0.5).astype(float))
    f1_test = f1_score(all_targets, (all_predictions > 0.5).astype(float), average='macro')
    
    return f'Test Loss: {total_loss:.4f}, Test Accuracy: {accuracy_test:.4f}, Test F1 Score: {f1_test:.4f}'

In [89]:
torch.cuda.empty_cache()

In [90]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}{'-'*130}")
    train_metrics = train(model, train_loader, optimizer, criterion, device)
    test_metrics = test(model, test_loader, criterion, device)
    scheduler.step()
    
    print(train_metrics)
    print(test_metrics)
    torch.cuda.empty_cache()

Epoch 1/3----------------------------------------------------------------------------------------------------------------------------------


100%|██████████| 125/125 [00:11<00:00, 11.09it/s]


Train Loss: 138.9298, Train Accuracy: 0.8786, Train F1 Score: 0.8841
Test Loss: 27.6797, Test Accuracy: 0.9127, Test F1 Score: 0.9127
Epoch 2/3----------------------------------------------------------------------------------------------------------------------------------


100%|██████████| 125/125 [00:11<00:00, 11.11it/s]


Train Loss: 63.2460, Train Accuracy: 0.9549, Train F1 Score: 0.9559
Test Loss: 35.5765, Test Accuracy: 0.9075, Test F1 Score: 0.9075
Epoch 3/3----------------------------------------------------------------------------------------------------------------------------------


100%|██████████| 125/125 [00:11<00:00, 11.12it/s]

Train Loss: 26.9435, Train Accuracy: 0.9836, Train F1 Score: 0.9840
Test Loss: 40.5258, Test Accuracy: 0.9083, Test F1 Score: 0.9087


In [92]:
PATH = f"weights/binary.pth"
torch.save(model.state_dict(), PATH)